In [7]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor
import json
from dotenv import load_dotenv
load_dotenv()

def load(path, to_drop):

    """
    The train and test sets are read from the data directory.
    The procedure returns the X matrix and y vector for training and testing phases.
    """

    df: pd.DataFrame = pd.read_csv(path)

    if "planningDate_dt" in df.columns:
        df["planningDate_dt"] = pd.to_datetime(df["planningDate_dt"])
        df = df.sort_values("planningDate_dt").reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    if df["org"].dtype == object: df["org"] = df["org"].map({"org6": 0, "org9": 1})

    cols = [c for c in to_drop if c in df.columns]
    df = df.drop(columns=cols)

    y = df["totalAssignments"] # the vector of the target variable

    # the matrix of dataset features
    if "planningDate_dt" in df.columns: X = df.drop(columns=["totalAssignments", "planningDate_dt"])
    else: X = df.drop(columns=["totalAssignments"])
    return X, y

def print_results(y_test, preds, X_test):

    """
    The procedure prints the results given by the trained Machine Learning models.
    The results include the regression metrics for the entire dataset and the ones obtained in the
    different orgs.
    """

    print("Results for the global dataset:")
    print(f"  MAE:  {mean_absolute_error(y_test, preds):.4f}")
    print(f"  RMSE: {root_mean_squared_error(y_test, preds):.4f}")
    print(f"  R2:   {r2_score(y_test, preds):.4f}")

    print("Results divided by different ORG:")

    orgs = {"org6": 0, "org9": 1}

    for org_name, org_code in orgs.items():
        row = (X_test["org"] == org_code).values # we firstly evaluate the ORG6 rows, ORG9 then
        if row.sum() == 0: continue

        y_real = y_test.values[row]
        y_pred = preds[row]

        mae = mean_absolute_error(y_real, y_pred)
        rmse = root_mean_squared_error(y_real, y_pred)
        r2 = r2_score(y_real, y_pred)

        print(f"  [{org_name}] MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

def train_and_evaluate(model, param_grid, X_train, y_train, X_test, y_test, cv_splitter, model_name):

    """
    The Machine Learning models are trained and evaluated on real data.
    The results are printed by the utility function declared above.

    We firstly perform a tuning phase, using Randomized Search, to find the best
    hyperparameters for the model. Then, the best estimator is evaluated.
    """

    print(f"\n--- {model_name} Training ---")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=cv_splitter,
        n_iter=50,
        n_jobs=-1,
        scoring="neg_mean_squared_error",
        random_state=42
    )
    search.fit(X_train, y_train)

    print("Tuning finished. Best Params are the following:")
    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")

    best_model = search.best_estimator_
    preds = best_model.predict(X_test)

    print_results(y_test, preds, X_test)
    return best_model, preds

def train_tabpfn(X_train, y_train, X_test, y_test):

    """
    The TabPFN model is evaluated on real data.
    No tuning is needed as the foundation model is already pre-trained on synthetic tabular data.
    """

    print("\n--- TabPFN Training ---")

    token = os.environ.get("TABPFN_TOKEN")
    if not token:
        raise EnvironmentError()

    model = TabPFNRegressor()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print_results(y_test, preds, X_test)
    return model, preds

def baseline_model(X_train, y_train, X_test, y_test):

    """
    To evaluate the performance of Machine Learning we built a stupid baseline model.
    This model simply calculate the rounded mean of the totalAssignments of the y vector for both orgs.

    """
    org_train = X_train["org"].values
    org_test = X_test["org"].values

    mean_org6 = round(y_train.values[org_train == 0].mean(), 1)
    mean_org9 = round(y_train.values[org_train == 1].mean(), 1)
    # print("MEAN FOR ORG6:", mean_org6, "| MEAN FOR ORG9:", mean_org9)

    baseline_preds = np.where(org_test == 0, mean_org6, mean_org9)
    print("\n--- Baseline ---")
    print_results(y_test, baseline_preds, X_test)


# MAIN
if __name__ == "__main__":

    # In EDA phase, some columns were dropped for target leakage
    # (r >= 0.95) and multicollinearity between features.
    # This selection was decided using the train set only, to avoid leaking information from the test set.
    # The decision is stored in a JSON so it can be applied to both train and test in the load function
    with open("data/output/columns_to_be_dropped.json") as f:
        columns_to_be_dropped = json.load(f)

    X_train, y_train = load("data/output/refactored_train.csv", columns_to_be_dropped)
    X_test, y_test = load("data/output/test.csv", columns_to_be_dropped)


    time_split = TimeSeriesSplit(n_splits=3) # Cross Validation to avoid temporal data leakage

    # Params that the Randomized Search will evaluate for the Random Forest model
    rf_params = {
        "n_estimators": [50, 100, 150, 200, 250, 300],
        "max_depth": [5, 8, 12, 15],
        "min_samples_split": [2, 4, 6, 8],
        "max_features": ["sqrt", "log2", 0.5],
    }
    rf_model, rf_preds = train_and_evaluate(RandomForestRegressor(random_state=42, n_jobs=-1),
        rf_params, X_train, y_train, X_test, y_test, time_split, "Random Forest")

    # Params that the Randomized Search will evaluate for the XgBoost model
    xgb_params = {
        "n_estimators": [50, 100, 150, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7, 9],
        "subsample": [0.4, 0.6, 0.8, 1.0],
        "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
        "reg_lambda": [1, 5, 10, 15],
    }
    xgb_model, xgb_preds = train_and_evaluate(XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror"),
        xgb_params, X_train, y_train, X_test, y_test, time_split, "XGBoost")

    tabpfn_model, tabpfn_preds = train_tabpfn(X_train, y_train, X_test, y_test)

    baseline_model(X_train, y_train, X_test, y_test)


--- Random Forest Training ---
Tuning finished. Best Params are the following:
  n_estimators: 50
  min_samples_split: 4
  max_features: sqrt
  max_depth: 15
Results for the global dataset:
  MAE:  2.9969
  RMSE: 3.7519
  R2:   0.9712
Results divided by different ORG:
  [org6] MAE=3.1467  RMSE=4.0969  R2=-1.4453
  [org9] MAE=2.8472  RMSE=3.3718  R2=-0.6200

--- XGBoost Training ---
Tuning finished. Best Params are the following:
  subsample: 1.0
  reg_lambda: 15
  n_estimators: 200
  max_depth: 3
  learning_rate: 0.1
  colsample_bytree: 0.8
Results for the global dataset:
  MAE:  2.2599
  RMSE: 2.8676
  R2:   0.9832
Results divided by different ORG:
  [org6] MAE=2.5302  RMSE=3.2742  R2=-0.5619
  [org9] MAE=1.9897  RMSE=2.3927  R2=0.1842

--- TabPFN Training ---
Results for the global dataset:
  MAE:  0.9683
  RMSE: 1.2800
  R2:   0.9967
Results divided by different ORG:
  [org6] MAE=1.3692  RMSE=1.6554  R2=0.6008
  [org9] MAE=0.5675  RMSE=0.7325  R2=0.9236

--- Baseline ---
Results fo